# Natural Language Processing - Text Preprocessing

## Libraries and settings

In [16]:
# Libraries
import os
import re
import string
import numpy as np
import pandas as pd
from pprint import pprint

import nltk

# Import only once
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger')

from nltk.tag import pos_tag
from nltk.corpus import stopwords
from nltk.chunk import tree2conlltags
from nltk.chunk import conlltags2tree
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Current working directory
print('Current working directory:', os.getcwd())

Current working directory: /workspaces/data_analytics/Week_11


[nltk_data] Downloading package stopwords to /home/vscode/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /home/vscode/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/vscode/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /home/vscode/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/vscode/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


## Defining documents

In [17]:
# Defining documents (=sentenses)
d1 = 'Football is played on the field.'
d2 = 'Basketball is played on the court.'
d3 = 'Tennis is played on the tennis court.'

corpus_01 = d1 + ' ' + d2 + ' ' + d3
corpus_01

'Football is played on the field. Basketball is played on the court. Tennis is played on the tennis court.'

## Text preprocessing
#### Steps:
- Text to lowercase
- Removing punctuations
- Tokenization
- Removal of stop words
- Lemmatization

### Text to lowercase

In [18]:
# Text to lowercase function
def text_lowercase(text):
    return text.lower()

# Text to lowercase
corpus_02 = text_lowercase(corpus_01)
corpus_02

'football is played on the field. basketball is played on the court. tennis is played on the tennis court.'

### Removing punctuation

In [19]:
# Remove punctuation function
def remove_punctuation(text):
    translator = str.maketrans('', '', string.punctuation)
    return text.translate(translator)

# Remove punctuation
corpus_03 = remove_punctuation(corpus_02)
corpus_03

'football is played on the field basketball is played on the court tennis is played on the tennis court'

### Tokenize text & removal of stopwords

In [20]:
# Show english stopwords
eng_stopwords = set(stopwords.words('english'))
print("List of english stopwords:")
print(eng_stopwords)

List of english stopwords:
{'was', 'doesn', "wouldn't", "should've", 'until', 'aren', "i'm", 'for', "he'd", 'he', 've', 'by', 'herself', 'while', 'such', "haven't", 'few', 'itself', 'no', 'mustn', "he's", 'more', 'mightn', 'out', 'why', "it'll", 'myself', "she'd", 'under', 'your', 'and', 'their', 'it', 'haven', 'there', "shan't", "it's", "they've", 'own', 'too', 'wouldn', "we'd", 'above', "i'll", 'about', 'you', "it'd", 'him', 'hadn', 'shan', 'do', 'on', 'himself', 'down', 'into', 'through', 'me', 'most', 'his', "mightn't", "we've", 'won', "you'll", 'will', 'than', 'with', 'these', "don't", "isn't", "they'd", 'in', "she'll", "didn't", 'don', 'only', 'after', 'nor', 'does', 'she', "that'll", 'my', 'up', 'having', 'further', 'or', "wasn't", 'hasn', 'this', 'are', 'both', 'here', 'been', "she's", 'we', 'its', 'is', 'where', 'same', 'as', 'to', 'but', "won't", "mustn't", "i'd", 's', 'm', 'did', "hasn't", 'be', 'how', "i've", 'then', 'again', 'yourselves', 'during', 'isn', 'had', 'couldn', 

In [21]:
# Function for tokenization and the removal of stopwords
def remove_stopwords(text):
    stop_words = set(stopwords.words("english"))
    word_tokens = word_tokenize(text)
    filtered_text = [word for word in word_tokens if word not in stop_words]
    return filtered_text
 
# Remove stopwords
corpus_04 = remove_stopwords(corpus_03)
print(corpus_04, end="")

['football', 'played', 'field', 'basketball', 'played', 'court', 'tennis', 'played', 'tennis', 'court']

### Lemmatization

In [22]:
# Initialize Lemmatizer
lemmatizer = WordNetLemmatizer()

# Lemmatize string function
def lemmatize_word(text):
    word_tokens = word_tokenize(text)
    lemmas = [lemmatizer.lemmatize(word, pos ='v') for word in word_tokens]
    return lemmas

# Lemmatize
lem = []
for i in corpus_04:
    lem.append(lemmatize_word(i))

# Nested list to list
corpus_05 = [' '.join([str(x) for x in lst]) for lst in lem]

print('Before lemmatization:')
print(corpus_04, '\n')

print('After lemmatization:')
print(corpus_05, end="")

Before lemmatization:
['football', 'played', 'field', 'basketball', 'played', 'court', 'tennis', 'played', 'tennis', 'court'] 

After lemmatization:
['football', 'play', 'field', 'basketball', 'play', 'court', 'tennis', 'play', 'tennis', 'court']

## Redefine the text corpus (pre-processed)

In [23]:
# We will use the lemmatized words above to re-define our corpus 
corpus = ['football play field', 
          'basketball play court', 
          'tennis play tennis court']

## Document-term matrix with ngram_range=(1,1)

In [24]:
# Vectorizer with ngram_range=(1,1)
vectorizer = CountVectorizer(min_df=0.0, ngram_range=(1,1))

# Transform 
count = vectorizer.fit_transform(corpus)
 
# Create dataframe
df_count = pd.DataFrame(count.toarray(),
                        columns=vectorizer.get_feature_names_out())

print('Document-term matrix')
print(df_count)

Document-term matrix
   basketball  court  field  football  play  tennis
0           0      0      1         1     1       0
1           1      1      0         0     1       0
2           0      1      0         0     1       2


## Document-term matrix with ngram_range=(2,2)

In [25]:
# Vectorizer with with ngram_range=(2,2)
vectorizer = CountVectorizer(min_df=0.0, ngram_range=(2,2))

# Transform 
count = vectorizer.fit_transform(corpus)
 
# Create dataframe
df_count = pd.DataFrame(count.toarray(),
                        columns=vectorizer.get_feature_names_out())

print('Document-term matrix')
print(df_count)

Document-term matrix
   basketball play  football play  play court  play field  play tennis  \
0                0              1           0           1            0   
1                1              0           1           0            0   
2                0              0           0           0            1   

   tennis court  tennis play  
0             0            0  
1             0            0  
2             1            1  


## Term frequency-inverse document frequency (TF-IDF)
- For details see: https://www.learndatasci.com/glossary/tf-idf-term-frequency-inverse-document-frequency

### Term Frequency (TF)

In [26]:
# Compute Term Frequency (TF)
words_set = set()
for doc in corpus:
    words = doc.split(' ')
    words_set = words_set.union(set(words))
    
print('Number of words in the corpus:',len(words_set), '\n')
print('The words in the corpus: \n', words_set)

# Number of documents in the corpus
n_docs = len(corpus)

# Number of unique words in the corpus 
n_words_set = len(words_set)

df_tf = pd.DataFrame(np.zeros((n_docs, n_words_set)), 
                     columns=list(words_set))

print("\nTerm Frequency (TF):")
for i in range(n_docs):
    # Words in the document
    words = corpus[i].split(' ')
    for w in words:
        df_tf[w][i] = df_tf[w][i] + (1 / len(words))
        
print(df_tf.round(4))

Number of words in the corpus: 6 

The words in the corpus: 
 {'court', 'play', 'tennis', 'basketball', 'field', 'football'}

Term Frequency (TF):
    court    play  tennis  basketball   field  football
0  0.0000  0.3333     0.0      0.0000  0.3333    0.3333
1  0.3333  0.3333     0.0      0.3333  0.0000    0.0000
2  0.2500  0.2500     0.5      0.0000  0.0000    0.0000


### Inverse Document Frequency (IDF)

In [27]:
# Computing Inverse Document Frequency (IDF)
print("\nInverse Document Frequency (IDF):")

idf = {}

for w in words_set:
    
    # k = number of documents that contain this word
    k = 0
    
    for i in range(n_docs):
        if w in corpus[i].split():
            k += 1
            
    idf[w] =  np.log10(n_docs / k).round(4)
    
    print(f'{w:>15}: {idf[w]:>10}')


Inverse Document Frequency (IDF):
          court:     0.1761
           play:        0.0
         tennis:     0.4771
     basketball:     0.4771
          field:     0.4771
       football:     0.4771


### Term Frequency - Inverse Document Frequency (TF-IDF)

In [28]:
# Computing TF-IDF
df_tf_idf = df_tf.copy()

for w in words_set:
    for i in range(n_docs):
        df_tf_idf[w][i] = df_tf[w][i] * idf[w]

print('\nTF-IDF:')
print(df_tf_idf.round(4))


TF-IDF:
    court  play  tennis  basketball  field  football
0  0.0000   0.0  0.0000       0.000  0.159     0.159
1  0.0587   0.0  0.0000       0.159  0.000     0.000
2  0.0440   0.0  0.2386       0.000  0.000     0.000


## Part-of-Speach (POS) tagging
For meaning of POS-tags see: https://pythonexamples.org/nltk-pos-tagging

In [29]:
text = '''Regular strength training improves physical 
            health and increases overall quality of life.'''


def preprocess(sent):
    sent = nltk.word_tokenize(sent)
    sent = nltk.pos_tag(sent)
    return sent

sent = preprocess(text)
pattern = 'NP: {<DT>?<JJ>*<NN>}'

cp = nltk.RegexpParser(pattern)
cs = cp.parse(sent)

iob_tagged = tree2conlltags(cs)

# Print the POS-tags
pprint(iob_tagged)

[('Regular', 'JJ', 'B-NP'),
 ('strength', 'NN', 'I-NP'),
 ('training', 'NN', 'B-NP'),
 ('improves', 'VBZ', 'O'),
 ('physical', 'JJ', 'B-NP'),
 ('health', 'NN', 'I-NP'),
 ('and', 'CC', 'O'),
 ('increases', 'VBZ', 'O'),
 ('overall', 'JJ', 'B-NP'),
 ('quality', 'NN', 'I-NP'),
 ('of', 'IN', 'O'),
 ('life', 'NN', 'B-NP'),
 ('.', '.', 'O')]


JJ - Adjective: Describes or modifies a noun, providing additional information as *Regular*, *physical*, *overall*.

NN - Noun, singular: Represents a person, object, or abstract concept, such as *strength*, *training*, *health*, or *life*.

VBZ - Verb, third person singular present: Indicates an action in the present tense performed by a singular subject, such as *improves* or *increases*.

CC - Coordinating Conjunction: Connects words or clauses of equal importance, for example *and*.

IN - Preposition/Subortinating Conjunction: Expresses relationships between words, often indicating connection or inclusion, such as *of*.

### Jupyter notebook --footer info-- (please always provide this at the end of each submitted notebook)

In [30]:
import os
import platform
import socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.8.0-1030-azure
Datetime: 2025-12-13 20:00:16
Python Version: 3.11.14
-----------------------------------
